# 基于 WT-LightGBM 的用户侧日前出清电价预测系统

本 Notebook 演示如何从零开始使用本项目进行数据读取、特征构建、WT-LightGBM 模型训练和预测。

## 1. 项目结构

```
基于LightGBM的用户侧日前出清电价预测/
├── config/
│   ├── default.yaml              # 主配置文件
│   └── features.yaml             # 特征注册配置
├── Dataset/                      # 原始数据目录（CSV）
├── models/                       # 模型保存目录
├── data/                         # 输出数据目录
│   ├── features/                 # 特征矩阵
│   └── predictions/              # 预测结果
├── reports/                      # 评估报告与图像
├── src/                          # 源代码
│   ├── cli/main.py               # 命令行入口
│   ├── data/                     # 数据读取、校验、对齐
│   ├── features/                 # 特征工程
│   ├── models/                   # 模型实现
│   ├── pipeline/                 # 训练/预测流程
│   └── evaluation/               # 评估指标
├── tests/                        # 单元测试
├── requirements.txt              # Python 依赖
└── setup.py                      # 包安装脚本
```

## 2. 环境准备

### 2.1 安装依赖

In [ ]:
# 在 Jupyter 中安装依赖（已在环境中安装过可跳过）
# !pip install -r requirements.txt

### 2.2 导入常用库

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号

print("Python version:", sys.version)

### 2.3 运行单元测试验证环境

In [ ]:
!python -m pytest tests/ -q

预期输出：`18 passed`

## 3. 数据读取与探索

### 3.1 加载配置文件

In [ ]:
import yaml
from pathlib import Path

config_path = "config/default.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("数据粒度:", config["data"]["granularity"])
print("模型类型:", config["model"]["type"])
print("数据集根目录:", config["data"]["dataset_root"])

### 3.2 读取目标变量（电价）

In [ ]:
from src.data.reader import DatasetCSVReader

reader = DatasetCSVReader(
    dataset_root=config["data"]["dataset_root"],
    sources=config["data"]["sources"]
)

target_df = reader.read_target()
print("目标变量形状:", target_df.shape)
print("\n前 5 行:")
print(target_df.head())
print("\n时间范围:", target_df["timestamp"].min(), "~", target_df["timestamp"].max())

### 3.3 可视化电价走势

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(target_df["timestamp"], target_df["value"], alpha=0.7)
plt.title("用户侧日前出清电价走势")
plt.xlabel("时间")
plt.ylabel("电价")
plt.tight_layout()
plt.show()

### 3.4 读取特征表

In [ ]:
feature_names = [
    "sys_load_pred", "wind_power_pred", "solar_power_pred",
    "power_import_plan", "coal_gen_plan", "gas_gen_plan",
    "storage_plan", "reserve_pos", "reserve_neg", "renewable_capacity"
]

feature_tables = {}
for name in feature_names:
    if name in config["data"]["sources"]:
        feature_tables[name] = reader.read_table(name)
        print(f"{name}: {feature_tables[name].shape}")

## 4. 数据校验

In [ ]:
from src.data.validator import DataValidator

validator = DataValidator()
report = validator.validate(target_df, table_name="target_price")
print(report)

## 5. 特征工程

### 5.1 构建宽表面板

In [ ]:
from src.data.adapter import DataAdapter

adapter = DataAdapter(
    fill_00_with_24=config["preprocessing"].get("fill_00_with_24", True),
    solar_wind_night_fill=config["preprocessing"].get("solar_wind_night_fill", 0.0)
)

# 合并实际风光为 actual_wind_solar
actual_tables = {}
for name in ["actual_sys_load", "actual_wind", "actual_solar"]:
    if name in config["data"]["sources"]:
        actual_tables[name] = reader.read_table(name)

if "actual_wind" in actual_tables and "actual_solar" in actual_tables:
    combined = actual_tables["actual_wind"][["timestamp", "value"]].copy()
    combined = combined.merge(
        actual_tables["actual_solar"][["timestamp", "value"]].rename(columns={"value": "solar"}),
        on="timestamp", how="outer"
    )
    combined["value"] = combined["value"].fillna(0) + combined["solar"].fillna(0)
    combined["field_name"] = "actual_wind_solar"
    actual_tables["actual_wind_solar"] = combined[["timestamp", "value", "field_name"]]
    del actual_tables["actual_wind"]
    del actual_tables["actual_solar"]

all_tables = {**feature_tables, **actual_tables}
df = adapter.build_panel(all_tables, target_df, align_to_target=True)
print("面板数据形状:", df.shape)
print(df.head())

### 5.2 预处理

In [ ]:
from src.features.preprocessor import Preprocessor

preprocessor = Preprocessor(config["preprocessing"])
df = preprocessor.handle_missing(df, solar_wind_cols=["wind_power_pred", "solar_power_pred"])
print("预处理后缺失值统计:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

### 5.3 构建全部特征

In [ ]:
from src.features.feature_builder import FeatureBuilder
from src.features.feature_registry import FeatureRegistry
from src.utils.config import load_features_config

features_cfg = load_features_config(config["features"]["config_path"])
registry = FeatureRegistry.from_config(features_cfg)

builder = FeatureBuilder(
    lag_windows=config["features"].get("lag_windows", [1, 7]),
    rolling_windows=config["features"].get("rolling_windows", [96, 672]),
    registry=registry,
    use_chinese_calendar=config.get("calendar", {}).get("use_chinese_calendar", True)
)

features = builder.build_all(df)
features["target"] = df["target"].values
features = features.dropna()
print("特征矩阵形状:", features.shape)
print("\n特征列:")
print(features.columns.tolist())

## 6. 模型训练

### 6.1 划分训练/测试集

In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [c for c in features.columns if c not in ["timestamp", "target"]]
X = features[feature_cols].values
y = features["target"].values

# 按时间顺序划分，最后 30 天作为测试集
test_days = config["model"]["train"]["test_days"]
points_per_day = 96  # 15min 粒度
test_size = test_days * points_per_day

X_train, X_test = X[:-test_size], X[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

### 6.2 训练 WT-LightGBM 模型

In [ ]:
from src.models.wtlgbm_model import WTLGBMModel

model_config = {
    **config["model"].get("lgbm_params", {}),
    "decompose_level": config["model"].get("decompose_level", 2),
    "early_stopping_rounds": config["model"]["train"]["early_stopping_rounds"],
    "num_boost_round": config["model"]["train"]["num_boost_round"]
}

model = WTLGBMModel(model_config)
model.fit(X_train, y_train, X_test, y_test, feature_names=feature_cols)
print("模型训练完成")

### 6.3 测试集评估

In [ ]:
from src.evaluation.metrics import MetricsCalculator

y_pred = model.predict(X_test)
calc = MetricsCalculator()

# 按峰平谷时段分组
test_timestamps = features["timestamp"].iloc[-test_size:]
hours = pd.to_datetime(test_timestamps).dt.hour
periods = hours.apply(lambda h: 0 if h < 6 else (2 if 8 <= h < 11 or 17 <= h < 21 else 1)).values

metrics = calc.compute_all(y_test, y_pred, periods=periods)
for k, v in metrics.items():
    print(f"{k}: {v}")

### 6.4 保存模型

In [ ]:
import os

model_dir = Path(config["output"]["model_dir"])
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "wtlgbm_notebook.pkl"
model.save(str(model_path))
print(f"模型已保存至: {model_path}")

### 6.5 预测 vs 实际对比图

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_timestamps.values, y_test, label="真实值", alpha=0.8)
plt.plot(test_timestamps.values, y_pred, label="预测值", alpha=0.8)
plt.title("测试集预测结果对比")
plt.xlabel("时间")
plt.ylabel("电价")
plt.legend()
plt.tight_layout()
plt.show()

## 7. 单日期预测

### 7.1 使用预测流程预测 2026-06-30

In [ ]:
from src.pipeline.predict_pipeline import PredictPipeline

predict_pipeline = PredictPipeline(config_path="config/default.yaml")
result = predict_pipeline.run(target_date="2026-06-30", model_path=str(model_path))
print("预测结果前 10 行:")
print(result.head(10))
print(f"\n预测点数: {len(result)}")

### 7.2 可视化单日预测曲线

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(result["timestamp"], result["predicted_price"], marker="o", markersize=2)
plt.title("2026-06-30 日前出清电价预测（96 点）")
plt.xlabel("时间")
plt.ylabel("预测电价")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. 命令行方式（推荐批量使用）

```bash
# 训练
python -m src.cli.main --config config/default.yaml train

# 预测
python -m src.cli.main --config config/default.yaml predict --date 2026-06-30

# 评估
python -m src.cli.main --config config/default.yaml evaluate --date 2026-06-30

# 日报
python -m src.cli.main --config config/default.yaml daily --date 2026-06-30
```

## 9. 附录：小波分解可视化

In [ ]:
from src.models.wavelet import WaveletDecomposer

sample_signal = y_train[:96*7]  # 取 7 天数据做演示
decomposer = WaveletDecomposer(wavelet="db4", level=2)
components = decomposer.stationary_decompose(sample_signal)

fig, axes = plt.subplots(len(components)+1, 1, figsize=(14, 8), sharex=True)
axes[0].plot(sample_signal)
axes[0].set_title("原始信号")
for i, (key, comp) in enumerate(components.items()):
    axes[i+1].plot(comp)
    axes[i+1].set_title(f"分量: {key}")
plt.tight_layout()
plt.show()